In [2]:
# ==========================================
#Cell 1: Data Configuration
#Purpose: Setup the data.yaml file pointing to the dataset directories and defining the 3 classes.
# ==========================================

import yaml
import os

dataset_root = '/kaggle/input/dataset-for-final-training/Final_Dataset_for_Training'

# 2. Define the configuration content
config = {
    'path': dataset_root,       # Root path
    'train': 'train/images',    # Training images (relative to root)
    'val': 'val/images',        # Validation images
    'test': 'test/images',      # Test images
    'nc': 3,                    # Number of classes
    'names': ['Person', 'Fire', 'Smoke'] # Class names
}

# 3. Save the YAML file to the working directory
yaml_path = '/kaggle/working/data.yaml'

with open(yaml_path, 'w') as f:
    yaml.dump(config, f, sort_keys=False)

print(f"✅ Data config created successfully at:\n{yaml_path}")

✅ Data config created successfully at:
/kaggle/working/data.yaml


In [6]:
# ==========================================
# Cell 2: Setup and Fix Dependencies
# Purpose: Uninstall 'ray' to prevent AttributeError and install YOLO11.
# ==========================================

# 1. Uninstall 'ray' to fix the circular import error (The cause of your crash)
!pip uninstall -y ray ray-tune

# 2. Install Ultralytics
!pip install ultralytics

# 3. Import and disable Ray Tune integration explicitly
import ultralytics
ultralytics.settings.update({'raytune': False}) 

# 4. Check installation
ultralytics.checks()

Ultralytics 8.4.9 🚀 Python-3.12.12 torch-2.8.0+cu126 CUDA:0 (Tesla T4, 14913MiB)
Setup complete ✅ (4 CPUs, 31.4 GB RAM, 6634.1/8062.4 GB disk)


In [ ]:
# ==========================================
# Cell 3: Train YOLO11 Model
# Purpose: Initialize and train the YOLO11 model (Small version) on the custom dataset.
# ==========================================

from ultralytics import YOLO

# Settings
PROJECT_DIR = '/kaggle/working/training_result'
RUN_NAME = 'fire_smoke_v2'
YAML_FILE = '/kaggle/working/data.yaml' 

# Load the YOLO11 model (S = Small, better for features than Nano, faster than Medium)
# Note: Using 'yolo11s.pt' as requested for the new architecture
model = YOLO('yolo11s.pt') 

# Start Training
results = model.train(
    data=YAML_FILE,
    epochs=100,           
    patience=20,          # Stop if no improvement for 20 epochs
    batch=16,             
    imgsz=640,            
    project=PROJECT_DIR,  
    name=RUN_NAME,        
    exist_ok=True,        
    verbose=True,
    # Augmentation settings to help with smoke/fire texture
    mosaic=1.0,
    mixup=0.1
)

print(f"✅ Training finished. Best weights saved at: {PROJECT_DIR}/{RUN_NAME}/weights/best.pt")

In [8]:
# ====================================================================
# Cell 4: Detailed Validation (Split Metrics)
# Purpose: Evaluate the model and print specific performance metrics, 
#(mAP, Precision, Recall) separately for People vs. Fire/Smoke.
# ====================================================================

import numpy as np

# Load the best trained model
best_model = YOLO(f'{PROJECT_DIR}/{RUN_NAME}/weights/best.pt')

print("📊 Running Validation to get separated metrics...")
metrics = best_model.val(data=YAML_FILE, split='test', verbose=False)

# Extract metrics per class
# indices: 0=Person, 1=Fire, 2=Smoke
names = metrics.names
maps = metrics.box.maps   # mAP50-95 per class
p = metrics.box.p         # Precision per class
r = metrics.box.r         # Recall per class
ap50 = metrics.box.ap50   # mAP50 per class

print("\n" + "="*40)
print("🟢 PERFORMANCE ON PEOPLE (CRITICAL)")
print("="*40)
print(f"Class: {names[0]}")
print(f"Precision : {p[0]:.3f}")
print(f"Recall    : {r[0]:.3f}")
print(f"mAP50     : {ap50[0]:.3f}")

print("\n" + "="*40)
print("🔴 PERFORMANCE ON FIRE & SMOKE")
print("="*40)
for i in [1, 2]:
    print(f"Class: {names[i]}")
    print(f"  Precision : {p[i]:.3f}")
    print(f"  Recall    : {r[i]:.3f}")
    print(f"  mAP50     : {ap50[i]:.3f}")
    print("-" * 20)

📊 Running Validation to get separated metrics...
Ultralytics 8.4.9 🚀 Python-3.12.12 torch-2.8.0+cu126 CUDA:0 (Tesla T4, 14913MiB)
YOLO11s summary (fused): 101 layers, 9,413,961 parameters, 0 gradients, 21.3 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 61.9±15.9 MB/s, size: 503.1 KB)
val: Scanning /kaggle/input/dataset-for-final-training/Final_Dataset_for_Training/test/labels... 250 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 250/250 213.3it/s 1.2s0.0s
WARNING ⚠️ val: Cache directory /kaggle/input/dataset-for-final-training/Final_Dataset_for_Training/test is not writable, cache not saved.
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 16/16 4.1it/s 3.9s0.2s
                   all        250       2532      0.776      0.643      0.716      0.525
Speed: 1.3ms preprocess, 6.1ms inference, 0.0ms loss, 3.4ms postprocess per image
Results saved to /kaggle/working/runs/detect/val

🟢 PERFORMANCE ON PEOPLE (CRITICAL

In [10]:
# ===================================================================
# Cell 5: Clean, Zip, and Download (Consolidated)
# Purpose: Cleans up temporary zip files, zips the training results, and provides a direct download link for the model and graphs.
# ===================================================================

import shutil
import os
from IPython.display import FileLink

# 1. Cleanup old zips
work_dir = '/kaggle/working'
for f in os.listdir(work_dir):
    if f.endswith(".zip"):
        os.remove(os.path.join(work_dir, f))

# 2. Define paths
results_dir = '/kaggle/working/training_result'
output_name = 'Forest_Fire_YOLO11_Results'

# 3. Zip and Display
if os.path.exists(results_dir):
    print(f"📦 Compressing training results...")
    shutil.make_archive(output_name, 'zip', results_dir)
    
    print("✅ Done! Download your results (Model + Graphs) below:")
    display(FileLink(f'{output_name}.zip'))
else:
    print("❌ Training results directory not found.")

📦 Compressing training results...
✅ Done! Download your results (Model + Graphs) below:


/kaggle/working/Forest_Fire_YOLO11_Results.zip